Experiment - 1 1.Text-to-SQL Workflow


In [ ]:
!pip install -q openai

In [ ]:
import sqlite3
import os
import re
from openai import OpenAI

In [ ]:
from getpass import getpass
from openai import OpenAI

OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=OPENAI_API_KEY)

Enter your OpenAI API key: ··········


In [ ]:
!pip install -q transformers sentencepiece torch


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def generate_sql(question):

    prompt = f"""
Convert this natural language question into a SQLite SQL query.

Database schema:
students(
    id INTEGER,
    name TEXT,
    department TEXT,
    year INTEGER,
    score INTEGER
)

Rules:
- Use only the students table.
- Generate only SELECT queries.
- Do not explain.
- Return only the SQL query.

Question:
{question}

SQL:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    sql = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return sql.strip()

In [ ]:
question = "Which students scored above 80?"

sql = generate_sql(question)

print("Question:")
print(question)

print("\nGenerated SQL:")
print(sql)

Question:
Which students scored above 80?

Generated SQL:
INTEGER CLASS_Score_(__________________________________________________________________________________________


In [ ]:
def execute_sql(sql):
    try:
        cursor.execute(sql)
        return cursor.fetchall()

    except Exception as e:
        return f"SQL Error: {e}"

In [ ]:
def generate_sql(question):

    prompt = """
You are a SQL generator.

Convert the question into SQLite SQL.

Database:
students(
    id INTEGER,
    name TEXT,
    department TEXT,
    year INTEGER,
    score INTEGER
)

Examples:

Question: Show all students
SQL: SELECT * FROM students;

Question: Which students scored above 80?
SQL: SELECT name, score FROM students WHERE score > 80;

Question: Show students from CSE
SQL: SELECT * FROM students WHERE department = 'CSE';

Question: Who has the highest score?
SQL: SELECT name, score FROM students ORDER BY score DESC LIMIT 1;

Question: How many students are there?
SQL: SELECT COUNT(*) FROM students;

Now convert this question:

Question:
""" + question + """

SQL:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=80
    )

    sql = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return sql.strip()

In [ ]:
question = "Which students scored above 80?"

sql = generate_sql(question)

print("Question:")
print(question)

print("\nGenerated SQL:")
print(sql)

Question:
Which students scored above 80?

Generated SQL:
SELECT id, school_id FROM students WHERE school_id = '80'


In [ ]:
result = execute_sql(sql)

print("Database Result:")

if isinstance(result, str):
    print(result)
else:
    for row in result:
        print(row)

Database Result:
SQL Error: no such column: school_id


In [ ]:
print("Generated SQL:")
print(sql)

Generated SQL:
SELECT id, school_id FROM students WHERE school_id = '80'


In [ ]:
def validate_sql(sql):

    allowed_columns = {
        "id",
        "name",
        "department",
        "year",
        "score"
    }

    # Check whether the SQL contains school_id
    if "school_id" in sql.lower():
        return False, "Invalid column: school_id"

    # Only allow SELECT
    if not sql.strip().lower().startswith("select"):
        return False, "Only SELECT queries are allowed"

    return True, "SQL is valid"

In [ ]:
valid, message = validate_sql(sql)

print("Validation:", message)

if valid:
    result = execute_sql(sql)

    print("\nDatabase Result:")
    for row in result:
        print(row)

else:
    print("\nSQL was rejected.")

Validation: Invalid column: school_id

SQL was rejected.


OUTPUT

In [ ]:
sql = """
SELECT name, score
FROM students
WHERE score > 80;
"""

print("SQL:")
print(sql)

valid, message = validate_sql(sql)

print("\nValidation:", message)

if valid:
    result = execute_sql(sql)

    print("\nDatabase Result:")

    for row in result:
        print(row)

SQL:

SELECT name, score
FROM students
WHERE score > 80;


Validation: SQL is valid

Database Result:
('Anil', 88)
('Priya', 92)
('Sita', 81)


Experiment -2  RAG-Based Question Answering System

In [ ]:
documents = [
    """
    Internet of Things (IoT) is a system of physical devices connected
    to a network. These devices can collect, exchange and process data.
    IoT systems commonly use sensors, controllers, communication networks
    and applications.
    """,

    """
    ESP32 is a low-cost microcontroller commonly used in IoT projects.
    It provides processing capability and built-in wireless connectivity
    such as Wi-Fi and Bluetooth. ESP32 is useful for connected sensors,
    automation and monitoring applications.
    """,

    """
    Sensors are devices that measure physical or environmental properties.
    Examples include temperature, humidity, distance, light, motion and
    gas sensors. Sensor readings can be processed by a controller.
    """,

    """
    IoT automation combines sensing, processing and actuation.
    For example, a smart water-level system can measure the water level
    using an ultrasonic sensor and automatically control a water pump.
    """,

    """
    Cloud platforms can store and visualize IoT sensor data.
    They can provide dashboards, data analysis and alerts when sensor
    values cross predefined thresholds.
    """
]

print("Number of documents:", len(documents))

Number of documents: 5


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english"
)

document_vectors = vectorizer.fit_transform(documents)

print("Documents indexed successfully!")
print("Index shape:", document_vectors.shape)

Documents indexed successfully!
Index shape: (5, 76)


In [ ]:
def retrieve_documents(question, top_k=3):

    # Convert question into a vector
    question_vector = vectorizer.transform([question])

    # Calculate similarity
    similarities = cosine_similarity(
        question_vector,
        document_vectors
    )[0]

    # Get highest similarity indexes
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "document": documents[index],
            "score": similarities[index]
        })

    return results

In [ ]:
question = "What does ESP32 provide for IoT projects?"

results = retrieve_documents(question)

for i, result in enumerate(results):

    print("=" * 60)
    print("Document:", i + 1)
    print("Similarity Score:", round(result["score"], 4))
    print(result["document"])

Document: 1
Similarity Score: 0.3796

    ESP32 is a low-cost microcontroller commonly used in IoT projects.
    It provides processing capability and built-in wireless connectivity
    such as Wi-Fi and Bluetooth. ESP32 is useful for connected sensors,
    automation and monitoring applications.
    
Document: 2
Similarity Score: 0.1769

    Cloud platforms can store and visualize IoT sensor data.
    They can provide dashboards, data analysis and alerts when sensor
    values cross predefined thresholds.
    
Document: 3
Similarity Score: 0.0809

    Internet of Things (IoT) is a system of physical devices connected
    to a network. These devices can collect, exchange and process data.
    IoT systems commonly use sensors, controllers, communication networks
    and applications.
    


In [ ]:
def create_context(question, top_k=3):

    results = retrieve_documents(question, top_k)

    context = ""

    for i, result in enumerate(results):

        context += f"""
Document {i + 1}:

{result["document"]}

"""

    return context

In [ ]:
question = "What does ESP32 provide for IoT projects?"

context = create_context(question)

print(context)


Document 1:


    ESP32 is a low-cost microcontroller commonly used in IoT projects.
    It provides processing capability and built-in wireless connectivity
    such as Wi-Fi and Bluetooth. ESP32 is useful for connected sensors,
    automation and monitoring applications.
    


Document 2:


    Cloud platforms can store and visualize IoT sensor data.
    They can provide dashboards, data analysis and alerts when sensor
    values cross predefined thresholds.
    


Document 3:


    Internet of Things (IoT) is a system of physical devices connected
    to a network. These devices can collect, exchange and process data.
    IoT systems commonly use sensors, controllers, communication networks
    and applications.
    




In [ ]:
def rag_answer(question):

    # Step 1: Retrieve relevant documents
    context = create_context(question)

    # Step 2: Create prompt
    prompt = f"""
Answer the question using only the information in the documents.

Documents:
{context}

Question:
{question}

Answer:
"""

    # Step 3: Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    # Step 4: Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    # Step 5: Decode
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
question = "What does ESP32 provide for IoT projects?"

answer = rag_answer(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What does ESP32 provide for IoT projects?

Answer:
processing capability and built-in wireless connectivity


OUTPUT


In [ ]:
question = "How can IoT automatically control a water pump?"

print("=" * 70)
print("USER QUESTION")
print("=" * 70)
print(question)

results = retrieve_documents(question)

print("\n" + "=" * 70)
print("RETRIEVED DOCUMENTS")
print("=" * 70)

for i, result in enumerate(results):

    print(f"\nDocument {i + 1}")
    print("Similarity:", round(result["score"], 4))
    print(result["document"])

answer = rag_answer(question)

print("\n" + "=" * 70)
print("GENERATED ANSWER")
print("=" * 70)
print(answer)

USER QUESTION
How can IoT automatically control a water pump?

RETRIEVED DOCUMENTS

Document 1
Similarity: 0.5995

    IoT automation combines sensing, processing and actuation.
    For example, a smart water-level system can measure the water level
    using an ultrasonic sensor and automatically control a water pump.
    

Document 2
Similarity: 0.0709

    Internet of Things (IoT) is a system of physical devices connected
    to a network. These devices can collect, exchange and process data.
    IoT systems commonly use sensors, controllers, communication networks
    and applications.
    

Document 3
Similarity: 0.0374

    Cloud platforms can store and visualize IoT sensor data.
    They can provide dashboards, data analysis and alerts when sensor
    values cross predefined thresholds.
    

GENERATED ANSWER
combines sensing, processing and actuation


Experiment - 3 3.Prompt Chaining for Summarization

In [ ]:
text = """
Internet of Things (IoT) is a technology that connects physical devices
to the internet so that they can collect and exchange data. IoT systems
usually contain sensors, controllers, communication networks and
applications.

Sensors can measure temperature, humidity, distance, motion, light and
gas concentration. The collected data is sent to a controller such as
an ESP32. The controller can process the data and make decisions based
on predefined conditions.

ESP32 is widely used in IoT projects because it provides processing
capability along with Wi-Fi and Bluetooth connectivity. It can connect
sensors to cloud platforms and other devices.

IoT automation can be used in smart homes, smart agriculture, smart
factories and water-management systems. For example, a smart water
system can measure the water level of a tank and automatically switch
a pump ON or OFF.

Cloud platforms can store and visualize sensor data. They can also
generate alerts when sensor readings cross predefined thresholds.
Therefore, IoT can help organizations monitor systems, automate
processes and make decisions using real-time data.
"""

print(text)


Internet of Things (IoT) is a technology that connects physical devices
to the internet so that they can collect and exchange data. IoT systems
usually contain sensors, controllers, communication networks and
applications.

Sensors can measure temperature, humidity, distance, motion, light and
gas concentration. The collected data is sent to a controller such as
an ESP32. The controller can process the data and make decisions based
on predefined conditions.

ESP32 is widely used in IoT projects because it provides processing
capability along with Wi-Fi and Bluetooth connectivity. It can connect
sensors to cloud platforms and other devices.

IoT automation can be used in smart homes, smart agriculture, smart
factories and water-management systems. For example, a smart water
system can measure the water level of a tank and automatically switch
a pump ON or OFF.

Cloud platforms can store and visualize sensor data. They can also
generate alerts when sensor readings cross predefined thres

In [ ]:
def generate_text(prompt, max_tokens=120):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result.strip()

In [ ]:
prompt1 = f"""
Read the following text and identify the most important points.

TEXT:
{text}

Important points:
"""

key_points = generate_text(prompt1, 150)

print("STAGE 1 — KEY POINTS")
print("=" * 60)
print(key_points)

STAGE 1 — KEY POINTS
IoT can help organizations monitor systems, automate processes and make decisions using real-time data.


In [ ]:
prompt2 = f"""
Create a short summary using the following key points.

KEY POINTS:
{key_points}

Write a clear and concise draft summary:
"""

draft_summary = generate_text(prompt2, 120)

print("STAGE 2 — DRAFT SUMMARY")
print("=" * 60)
print(draft_summary)

STAGE 2 — DRAFT SUMMARY
IoT can help organizations monitor systems, automate processes and make decisions using real-time data.


In [ ]:
prompt3 = f"""
Improve the following summary.

Requirements:
- Make it clear.
- Remove unnecessary repetition.
- Keep the important information.
- Use simple English.
- Do not add new facts.

DRAFT SUMMARY:
{draft_summary}

FINAL SUMMARY:
"""

final_summary = generate_text(prompt3, 120)

print("STAGE 3 — FINAL SUMMARY")
print("=" * 60)
print(final_summary)

STAGE 3 — FINAL SUMMARY
IoT can help organizations monitor systems, automate processes and make decisions using real-time data.


OUT PUT


In [ ]:
print("=" * 70)
print("PROMPT CHAINING FOR SUMMARIZATION")
print("=" * 70)

print("\nORIGINAL TEXT")
print("-" * 70)
print(text)

print("\nSTAGE 1 — KEY POINT EXTRACTION")
print("-" * 70)
print(key_points)

print("\nSTAGE 2 — DRAFT SUMMARY")
print("-" * 70)
print(draft_summary)

print("\nSTAGE 3 — FINAL SUMMARY")
print("-" * 70)
print(final_summary)

PROMPT CHAINING FOR SUMMARIZATION

ORIGINAL TEXT
----------------------------------------------------------------------

Internet of Things (IoT) is a technology that connects physical devices
to the internet so that they can collect and exchange data. IoT systems
usually contain sensors, controllers, communication networks and
applications.

Sensors can measure temperature, humidity, distance, motion, light and
gas concentration. The collected data is sent to a controller such as
an ESP32. The controller can process the data and make decisions based
on predefined conditions.

ESP32 is widely used in IoT projects because it provides processing
capability along with Wi-Fi and Bluetooth connectivity. It can connect
sensors to cloud platforms and other devices.

IoT automation can be used in smart homes, smart agriculture, smart
factories and water-management systems. For example, a smart water
system can measure the water level of a tank and automatically switch
a pump ON or OFF.

Cloud 